# Entraînement et évaluation

Une séparation stratifiée réserve 20 % des données au test, avec une graine fixe. Les encodeurs et le normaliseur sont ajustés uniquement sur l’entraînement. Aucune donnée de test ne sert au rééquilibrage ; la forêt utilise des poids de classe.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd() if (Path.cwd() / 'app.py').exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from app import CATEGORIES, NUMERIC, preprocess_new_data, validate_data

def prepare_training_data(frame):
    if 'Churn' not in frame.columns:
        raise ValueError('The training dataset must include Churn.')
    frame = frame.copy()
    frame['TotalCharges'] = pd.to_numeric(frame['TotalCharges'], errors='coerce')
    frame = frame.dropna(subset=['TotalCharges'])
    target = frame['Churn'].astype('string').str.strip().map({'No': 0, 'Yes': 1, '0': 0, '1': 1})
    if target.isna().any() or target.nunique() != 2:
        raise ValueError('Churn must contain both No and Yes classes.')
    return validate_data(frame, max_rows=None), target.astype(int)

def train_model(frame, random_state=42):
    features, target = prepare_training_data(frame)
    train_x, test_x, train_y, test_y = train_test_split(
        features, target, test_size=0.2, random_state=random_state, stratify=target,
    )
    encoders = {column: LabelEncoder().fit(train_x[column]) for column in CATEGORIES}
    scaler = StandardScaler().fit(train_x[NUMERIC])
    model = RandomForestClassifier(
        n_estimators=200, max_depth=9, class_weight='balanced', random_state=random_state,
    )
    model.fit(preprocess_new_data(train_x, encoders, scaler), train_y)
    transformed_test = preprocess_new_data(test_x, encoders, scaler)
    predictions = model.predict(transformed_test)
    probabilities = model.predict_proba(transformed_test)[:, list(model.classes_).index(1)]
    metrics = {
        'model': 'RandomForestClassifier',
        'sklearn_version': sklearn.__version__,
        'random_state': random_state,
        'rows_received': len(frame),
        'rows_used': len(features),
        'rows_excluded_missing_total_charges': len(frame) - len(features),
        'train_rows': len(train_x),
        'test_rows': len(test_x),
        'accuracy': accuracy_score(test_y, predictions),
        'precision': precision_score(test_y, predictions, zero_division=0),
        'recall': recall_score(test_y, predictions, zero_division=0),
        'f1': f1_score(test_y, predictions, zero_division=0),
        'roc_auc': roc_auc_score(test_y, probabilities),
        'confusion_matrix': confusion_matrix(test_y, predictions, labels=[0, 1]).tolist(),
    }
    return model, encoders, scaler, metrics

df = pd.read_csv(ROOT / "data/telco_customers.csv")


In [ ]:
model, encoders, scaler, metrics = train_model(df)
pd.Series(metrics).to_frame("Evaluation")


## Matrice de confusion

Les lignes représentent la classe réelle et les colonnes la classe prédite. La classe positive est Churn.


In [ ]:
pd.DataFrame(metrics["confusion_matrix"], index=["Actual Not Churn", "Actual Churn"], columns=["Predicted Not Churn", "Predicted Churn"])


## Importance des variables

Ces importances décrivent le modèle ; elles ne prouvent pas une relation causale.


In [ ]:
pd.Series(model.feature_importances_, index=model.feature_names_in_).sort_values(ascending=False).to_frame("Importance")


Réexécuter les cellules de ce notebook pour entraîner et évaluer une nouvelle forêt. Les résultats restent en mémoire ; les modèles intégrés à l’application ne sont pas remplacés. Cette évaluation concerne le modèle entraîné ici, pas les artefacts historiques fournis.
